<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת 3.3: רגרסיה, מה יהיה הציון הכרוני?</p>

<p dir="rtl" style="direction:rtl; text-align:right;">זהו המשך מודרך של חלק ג במפגש 3. עכשיו המטרה היא <b>מספר</b>: הציון הכרוני <code dir="ltr">fma_sum_chr</code> (<span dir="ltr">FMA-UE</span>, 0 עד 66). נאמן שני מודלים, רגרסיה לינארית ו-K-NN, ננבא את מטופלי הבדיקה, ונשווה את גודל השגיאה בנקודות <span dir="ltr">FMA</span>. בסוף ננבא גם את הציון של מרים (דמות בדויה). משך מוערך: כ-20 דקות. בכל תא פקודה אחת שמדפיסה תוצאה אחת.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>כך עובד כל תא בדיקה:</b> אם התשובה נכונה יודפס <code dir="ltr">OK</code>. אם לא, תופיע שגיאה, וזה בסדר: תקנו והריצו שוב. כל מקום שבו כתוב <code dir="ltr">___</code> מחכה להשלמה שלכם.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות (<code dir="ltr">tidymodels</code> ו-<code dir="ltr">kknn</code>) ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצו אותו פעם אחת בתחילת המפגש.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
# tidymodels + kknn: about 100 packages in all, 43 more than tidyverse alone. From the Posit binary
# repository there is no compiling, but the time on a fresh Colab session was NOT measured here
# (a rough guess is a few minutes). Run this cell first, before class starts.
needed  <- c("tidymodels", "kknn")   # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
suppressPackageStartupMessages(library(tidymodels))   # recipes, parsnip, rsample, yardstick, tune, ggplot2, dplyr ...
tidymodels_prefer()                                   # when two packages share a name, prefer the tidymodels one

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנה מוכנה מראש (ממחברת 3.1)</p>

<p dir="rtl" style="direction:rtl; text-align:right;">שישה תאים שחוזרים על מה שעשינו במחברת 3.1: טעינה, פיצול (אותו <code dir="ltr">set.seed(820)</code>) ומתכון. הפעם המטרה במתכון היא הציון הכרוני עצמו, ולכן <code dir="ltr">outcome_mild</code> כבר אינה בטבלה המוכנה. הריצו בלי לשנות.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")

In [ ]:
stroke <- load_course_data("stroke_clean") |>        # the cleaned table: one row per patient
  as_tibble() |>                                      # make it a tibble
  mutate(sex = factor(sex, levels = c("F", "M")),     # categories are stored as factors
         lesion_type = factor(lesion_type, levels = c("ischemic", "hemorrhagic", "mixed")))

stroke_ml <- stroke |>
  select(id, fma_sum_chr,                                   # id and the chronic score (the target)
         fma_sum_sub, fma_a_sub, fma_bc_sub,                # features known at the subacute visit ...
         age, sex, lesion_type, lesion_vol, tao_sub) |>     # ... as in chapter 8 of the book
  mutate(outcome_mild = factor(if_else(fma_sum_chr >= 43, "mild", "not_mild"),   # the categorical target
                               levels = c("mild", "not_mild")))

In [ ]:
set.seed(820)                                                 # fixed seed: everyone gets the same split
stroke_split <- initial_split(stroke_ml, prop = 0.75, strata = outcome_mild)   # 75% training, 25% test
stroke_train <- training(stroke_split)                        # the training table
stroke_test  <- testing(stroke_split)                         # the test table (locked until the end)

In [ ]:
stroke_rec <- recipe(fma_sum_chr ~ fma_sum_sub + age + sex + lesion_vol,
                     data = stroke_train) |>      # the plan: which columns, and which is the target
  step_impute_mode(sex) |>                         # fill a missing category with the most common one
  step_log(lesion_vol) |>                          # log of the lesion volume (it is very skewed)
  step_normalize(all_numeric_predictors()) |>      # z-score for every numeric predictor
  step_dummy(all_nominal_predictors())             # categories -> 0/1 columns

In [ ]:
stroke_rec_prep <- prep(stroke_rec)                           # learn the numbers from the training table
train_baked <- bake(stroke_rec_prep, new_data = stroke_train)   # apply to the training table
test_baked  <- bake(stroke_rec_prep, new_data = stroke_test)    # apply to the test table (same numbers)

In [ ]:
names(train_baked)   # the prepared columns: 4 predictors and the target

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 1: רגרסיה לינארית</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>רגרסיה לינארית</b> לומדת משוואה: ציון כרוני מנובא = מספר + שיפוע לכל מאפיין. המבנה ב-<code dir="ltr">tidymodels</code> זהה לסיווג: מפרט, <code dir="ltr">fit()</code>, <code dir="ltr">predict()</code>. הסוג הוא <code dir="ltr">linear_reg()</code>. דוגמה: המפרט והאימון.</p>

In [ ]:
lm_spec <- linear_reg()                                      # a linear regression
lm_fit  <- fit(lm_spec, fma_sum_chr ~ ., data = train_baked)   # target: the chronic score; features: all other columns
class(lm_fit)[1]

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> <code dir="ltr">predict()</code> של מודל רגרסיה מחזירה עמודה <code dir="ltr">.pred</code> (מספר). בנו את <code dir="ltr">lm_results</code>: האמת (<code dir="ltr">fma_sum_chr</code>) והניבוי, לכל מטופל בבדיקה.</p>

In [ ]:
lm_results <- test_baked |>
  select(fma_sum_chr) |>                              # the true chronic score
  bind_cols(predict(___, new_data = ___))             # TODO: the model, and the table to predict
lm_results |> head(4)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(lm_results) == 13, all(c("fma_sum_chr", ".pred") %in% names(lm_results)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 2: K-NN לרגרסיה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">אותו רעיון של "המטופלים הדומים", אבל במקום הצבעה, הניבוי הוא <b>ממוצע</b> הציונים הכרוניים של K השכנים. ההבדל בקוד הוא רק המצב: <code dir="ltr">set_mode("regression")</code>. דוגמה: המפרט.</p>

In [ ]:
knn_spec <- nearest_neighbor(neighbors = 5, weight_func = "rectangular") |>   # K = 5, equal weights
  set_mode("regression")                                                          # predict a number
knn_spec

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> אמנו את K-NN על האימון, באותה נוסחה כמו ברגרסיה הלינארית.</p>

In [ ]:
knn_fit <- fit(___, fma_sum_chr ~ ., data = ___)   # TODO: the spec and the training table

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(inherits(knn_fit, "model_fit"), nrow(predict(knn_fit, new_data = test_baked)) == 13)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> בנו את <code dir="ltr">knn_results</code> כמו <code dir="ltr">lm_results</code>, בלי רמזים.</p>

In [ ]:
# TODO: build knn_results exactly like lm_results, with knn_fit
knn_results <- ___

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(knn_results) == 13, all(c("fma_sum_chr", ".pred") %in% names(knn_results)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 3: RMSE, MAE ו-R² במילים</p>

<ul dir="rtl" style="direction:rtl; text-align:right;">
<li><b>RMSE</b> (שורש ממוצע ריבועי השגיאות): "גודל השגיאה הטיפוסי", בנקודות <span dir="ltr">FMA</span>. שגיאות גדולות נענשות חזק יותר.</li>
<li><b>MAE</b> (ממוצע השגיאה המוחלטת): השגיאה הממוצעת, בנקודות <span dir="ltr">FMA</span>. קל יותר להסביר למטופל.</li>
<li><b>R² (<code dir="ltr">rsq</code>)</b>: איזה חלק מהשונות בציונים המודל מסביר. 1 = הכול, 0 = כמו לנחש את הממוצע.</li>
</ul>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">metric_set()</code> בונה פונקציה שמחשבת את שלושתם. דוגמה לרגרסיה הלינארית על קבוצת הבדיקה:</p>

In [ ]:
reg_metrics <- metric_set(rmse, mae, rsq)                       # one function, three metrics
reg_metrics(lm_results, truth = fma_sum_chr, estimate = .pred)   # linear regression on the test set

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> חשבו את אותם מדדים ל-K-NN ושמרו ב-<code dir="ltr">knn_metrics</code>.</p>

In [ ]:
knn_metrics <- reg_metrics(___, truth = fma_sum_chr, estimate = .pred)   # TODO: the K-NN results table
knn_metrics

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(knn_metrics) == 3, all(c("rmse", "mae", "rsq") %in% knn_metrics$.metric))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> השוו את שני המודלים בטבלה אחת (הקוד נתון, השלימו רק את הטבלה השנייה). <code dir="ltr">bind_rows(.id = ...)</code> מדביקה טבלאות ורושמת מאיזו טבלה כל שורה, ו-<code dir="ltr">pivot_wider()</code> הופכת את המדדים לעמודות.</p>

In [ ]:
lm_metrics <- reg_metrics(lm_results, truth = fma_sum_chr, estimate = .pred)   # linear regression metrics
compare_reg <- bind_rows(linear = lm_metrics, knn = ___, .id = "model") |>   # TODO: the K-NN metrics table
  select(model, .metric, .estimate) |>
  pivot_wider(names_from = .metric, values_from = .estimate)                          # one row per model
compare_reg

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(compare_reg) == 2, all(c("rmse", "mae", "rsq") %in% names(compare_reg)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 4: ציון מנובא מול ציון אמיתי</p>

<p dir="rtl" style="direction:rtl; text-align:right;">הגרף הכי שימושי לרגרסיה: כל נקודה היא מטופל בבדיקה. הקו המקווקו הוא "ניבוי מושלם" (מנובא = אמיתי). ככל שהנקודות קרובות לקו, כך המודל מדויק. <code dir="ltr">coord_equal()</code> נותנת לשני הצירים אותה סקאלה. דוגמה: הרגרסיה הלינארית.</p>

In [ ]:
ggplot(lm_results, aes(x = fma_sum_chr, y = .pred)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey50") +   # perfect prediction
  geom_point(size = 2.5) +                                                          # one point per test patient
  coord_equal(xlim = c(0, 66), ylim = c(0, 66)) +                                   # same scale on both axes
  labs(title = "Linear regression", x = "Actual chronic FMA-UE", y = "Predicted chronic FMA-UE")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> צרו את אותו גרף ל-K-NN. שנו רק את הטבלה ואת הכותרת.</p>

In [ ]:
ggplot(___, aes(x = fma_sum_chr, y = .pred)) +                                      # TODO: the K-NN results table
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey50") +   # perfect prediction
  geom_point(size = 2.5) +                                                          # one point per test patient
  coord_equal(xlim = c(0, 66), ylim = c(0, 66)) +                                   # same scale on both axes
  labs(title = "K-NN (K = 5)", x = "Actual chronic FMA-UE", y = "Predicted chronic FMA-UE")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 5: מול ניחוש פשוט, ומה עם מרים?</p>

<p dir="rtl" style="direction:rtl; text-align:right;">מודל שווה משהו רק אם הוא טוב יותר מ<b>ניחוש פשוט</b>: לנבא לכולם את ממוצע הציונים הכרוניים <b>באימון</b>. הפונקציה <code dir="ltr">rep(x, n)</code> חוזרת על ערך <code dir="ltr">n</code> פעמים, ו-<code dir="ltr">rmse_vec()</code> מחשבת RMSE ישר משני וקטורים. דוגמה: הממוצע באימון.</p>

In [ ]:
mean_train <- mean(stroke_train$fma_sum_chr)   # the simple guess: the training mean
mean_train

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> חשבו את ה-RMSE של הניחוש הפשוט בקבוצת הבדיקה: האמת היא <code dir="ltr">test_baked&#36;fma_sum_chr</code>, והניבוי הוא <code dir="ltr">mean_train</code> שחוזר 13 פעמים.</p>

In [ ]:
baseline_rmse <- rmse_vec(truth = test_baked$fma_sum_chr,
                         estimate = rep(___, nrow(test_baked)))   # TODO: the simple guess
baseline_rmse

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(baseline_rmse - sqrt(mean((stroke_test$fma_sum_chr - mean(stroke_train$fma_sum_chr))^2))) < 1e-9)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">עכשיו מרים (דמות בדויה ממפגשים 1 ו-2): בת 68, אחרי שבץ איסכמי, <span dir="ltr">FMA-UE</span> תת-חריף 45. נפח הנגע (15) <b>הומצא להדגמה</b>. כדי לנבא אותה, צריך להעביר אותה דרך <b>אותו מתכון</b> (אותם מספרים מהאימון). הקוד נתון:</p>

In [ ]:
miriam <- tibble(fma_sum_sub = 45, age = 68,                                       # invented demo patient
                 sex = factor("F", levels = c("F", "M")),
                 lesion_vol = 15)
miriam_baked <- bake(stroke_rec_prep, new_data = miriam)   # the same recipe, the same training numbers
miriam_baked

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> נבאו את הציון הכרוני של מרים בשני המודלים (<code dir="ltr">predict()</code> על <code dir="ltr">miriam_baked</code>).</p>

In [ ]:
miriam_lm  <- predict(lm_fit,  new_data = miriam_baked)$.pred   # linear regression
miriam_knn <- predict(___, new_data = miriam_baked)$.pred       # TODO: the K-NN model
c(linear = miriam_lm, knn = miriam_knn)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(length(miriam_lm) == 1, length(miriam_knn) == 1, miriam_lm > 20, miriam_lm < 66)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">ניבוי בלי טווח הוא חצי תשובה. ברגרסיה לינארית אפשר לבקש <b>רווח ניבוי</b> (<code dir="ltr">type = "pred_int"</code>): הטווח שבו צפוי ליפול הציון של מטופל חדש כמוה, ברמת ביטחון של 95%.</p>

In [ ]:
predict(lm_fit, new_data = miriam_baked, type = "pred_int")   # a 95% prediction interval for Miriam

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> איזה מודל מדויק יותר בבדיקה, והאם הוא טוב יותר מהניחוש הפשוט (<code dir="ltr">baseline_rmse</code>)? מה הטווח של מרים אומר לקלינאי? זכרו שיש רק 13 מטופלים בבדיקה.</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>